# Kadane (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/Kadane_cpp.ipynb)

Chapter 18 — Recursive versus iterative algorithms. Runnable notebook of `Kadane`: the algorithm in C++, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/18-recursive-vs-iterative/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/Kadane_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/Kadane_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `FibRec` → `FibRec` (PascalCase).

## Kadane: a better iterative algorithm

**Kadane**$(a,n)$ — *Input*: an array $a[1..n]$ of $n\ge1$ integers. *Output*: the maximum sum of a non-empty subarray $a[l..r]$. It is the compact version of the chapter: $cur\leftarrow\max(a[i],\,cur+a[i])$, $best\leftarrow\max(best,cur)$. The loop makes $n-1$ iterations: $\Theta(n)$ steps and $\Theta(1)$ memory, against the $n(n+1)/2$ pairs $(l,r)$ of brute force ($\Theta(n^2)$) and against the recursive divide and conquer ($\Theta(n\log_2 n)$): a good iterative algorithm can beat recursion.

On the example of the price changes of chapter 17 the result is $43$. Then, on pseudo-random arrays with values between $-50$ and $50$ (same generator in all languages, seed $n$), the iterations of `Kadane`, the pairs examined by brute force and the check that the two algorithms find the same maximum sum; finally the measured times.

In [1]:
%%writefile kadane_compact.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Kadane(a, n): maximum sum of a non-empty subarray of a[0..n-1]
int Kadane(int *a, int n)
{
    int best = a[0];
    int cur = a[0];                 // maximum sum of a subarray ending at i
    int i = 1;
    while (i < n)
    {
        cur = max(a[i], cur + a[i]);
        best = max(best, cur);
        i = i + 1;
    }
    return best;
}

// Brute force: maximum sum and number of pairs (l, r) examined
int BruteForceBest(int *a, int n, long long &pairs)
{
    int best = a[0];
    pairs = 0;
    for (int l = 0; l < n; l++)
    {
        int s = 0;
        for (int r = l; r < n; r++)
        {
            s = s + a[r];
            pairs++;
            if (s > best) best = s;
        }
    }
    return best;
}

// Integers between -50 and 50 (linear congruential generator)
vector<int> RandomValues(int n, unsigned long long seed = 12345)
{
    unsigned long long x = seed;
    vector<int> a(n);
    for (int i = 0; i < n; i++)
    {
        x = (x * 1103515245ULL + 12345ULL) % 2147483648ULL;
        a[i] = (int)(x % 101ULL) - 50;
    }
    return a;
}

double NowMs()
{
    return chrono::duration<double, milli>(chrono::steady_clock::now().time_since_epoch()).count();
}

#ifndef NOMAIN
int main()
{
    int a[] = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
    printf("a = %s\n", Fmt(a, 16).c_str());
    printf("Kadane(a, 16) = %d\n", Kadane(a, 16));
    int b[] = {-3, -1, -2, -5};
    printf("a = %s  Kadane(a, 4) = %d\n", Fmt(b, 4).c_str(), Kadane(b, 4));
    printf("\n");
    printf("%6s %7s %11s %11s %9s  same sum?\n", "n", "Kadane", "BruteForce", "iterations", "pairs");
    vector<array<double, 3>> times;
    for (int n : {1000, 2000, 4000})
    {
        vector<int> v = RandomValues(n, n);
        long long pairs;
        double t0 = NowMs(); int k = Kadane(v.data(), n); double t1 = NowMs() - t0;
        t0 = NowMs(); int bf = BruteForceBest(v.data(), n, pairs); double t2 = NowMs() - t0;
        printf("%6d %7d %11d %11d %9lld  %s\n", n, k, bf, n - 1, pairs, k == bf ? "OK" : "ERROR");
        times.push_back({(double)n, t1, t2});
    }
    printf("\n");
    printf("Measured times (they depend on the machine):\n");
    for (auto &t : times)
    {
        printf("n=%-6d Kadane %9.4f ms   BruteForce %10.3f ms\n", (int)t[0], t[1], t[2]);
    }
    return 0;
}
#endif

Writing kadane_compact.cpp


In [2]:
!g++ -std=c++17 -O2 -o kadane_compact kadane_compact.cpp && ./kadane_compact

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
Kadane(a, 16) = 43
a = [-3, -1, -2, -5]  Kadane(a, 4) = -1

     n  Kadane  BruteForce  iterations     pairs  same sum?
  1000     547         547         999    500500  OK
  2000     981         981        1999   2001000  OK
  4000    1955        1955        3999   8002000  OK

Measured times (they depend on the machine):
n=1000   Kadane    0.0012 ms   BruteForce      0.375 ms
n=2000   Kadane    0.0023 ms   BruteForce      1.436 ms
n=4000   Kadane    0.0051 ms   BruteForce      6.082 ms


**Try it yourself.** Change $n$: the pairs of brute force grow as $n^2$, the iterations of Kadane as $n$.

In [3]:
%%writefile prova_kadane_compact.cpp
#define NOMAIN
#include "kadane_compact.cpp"

int main()
{
    // Try n = 8000: the pairs of brute force are n(n+1)/2 = 32 004 000, the iterations of Kadane 7999.
    int n = 8000;
    vector<int> v = RandomValues(n);
    long long pairs;
    int bf = BruteForceBest(v.data(), n, pairs);
    printf("%d (%d, %lld)\n", Kadane(v.data(), n), bf, pairs);
    return 0;
}

Writing prova_kadane_compact.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_kadane_compact prova_kadane_compact.cpp && ./prova_kadane_compact

1903 (1903, 32004000)


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).